# Tool calling

The model asks for a function; you run it and hand back the result.

*A brain in a jar — fluent, and unable to multiply or read a clock. The schema is a menu it
can point at; you are the hands.*

## 0. Setup

`cp ../.env.example ../.env`, add `DEEPINFRA_API_KEY`. DeepInfra speaks the OpenAI protocol.

In [1]:
import json
import os
from datetime import datetime
from zoneinfo import ZoneInfo

from dotenv import find_dotenv, load_dotenv
from openai import OpenAI

load_dotenv(find_dotenv(usecwd=True))
client = OpenAI(
    api_key=os.environ["DEEPINFRA_API_KEY"],
    base_url="https://api.deepinfra.com/v1/openai",
)
MODEL = "meta-llama/Meta-Llama-3.1-70B-Instruct-Turbo"

## 1. The gap

A model can't compute exactly, and can't know what time it is.

In [2]:
def ask(q):
    return client.chat.completions.create(
        model=MODEL, max_tokens=80, messages=[{"role": "user", "content": q}],
    ).choices[0].message.content.strip()


print("Q: 47281 * 918?")
print("  ", ask("What is 47281 * 918? Reply with only the number."))
print("   truth:", 47281 * 918, "\n")

print("Q: time in Bangkok?")
print("  ", ask("What time is it in Bangkok?")[:120], "...")

Q: 47281 * 918?
   43325938
   truth: 43403958 

Q: time in Bangkok?
   However, I'm a large language model, I don't have real-time access to the current time. But I can tell you that Bangkok, ...


## 2. The tools

Two halves that must agree: the function, and the schema — the only half the model sees.

In [3]:
def multiply(a: int, b: int) -> int:
    """Multiply two integers exactly."""
    return a * b


def current_time(timezone: str) -> str:
    """Wall-clock time in an IANA timezone."""
    return datetime.now(ZoneInfo(timezone)).strftime("%Y-%m-%d %H:%M:%S %Z")


TOOLS = [
    {"type": "function", "function": {
        "name": "multiply",
        "description": "Multiply two integers exactly.",
        "parameters": {"type": "object", "required": ["a", "b"], "properties": {
            "a": {"type": "integer", "description": "first factor"},
            "b": {"type": "integer", "description": "second factor"}}}}},
    {"type": "function", "function": {
        "name": "current_time",
        "description": "Current wall-clock time in an IANA timezone.",
        "parameters": {"type": "object", "required": ["timezone"], "properties": {
            "timezone": {"type": "string",
                         "description": "IANA name, e.g. 'Asia/Bangkok'"}}}}},
]

IMPLS = {"multiply": multiply, "current_time": current_time}   # schema name -> function

## 3. The loop

Ask → run what it asks for → hand back → repeat. No `tool_calls` means it's answering.

In [4]:
def run(question, max_steps=5, verbose=True):
    messages = [{"role": "user", "content": question}]

    for _ in range(max_steps):
        m = client.chat.completions.create(
            model=MODEL, max_tokens=512, tools=TOOLS, messages=messages,
        ).choices[0].message

        if not m.tool_calls:                       # no tool wanted -> final answer
            return (m.content or "").strip(), messages

        messages.append({"role": "assistant", "content": None,
                         "tool_calls": [tc.model_dump() for tc in m.tool_calls]})

        for tc in m.tool_calls:
            args = json.loads(tc.function.arguments)     # arguments arrive as a STRING
            result = IMPLS[tc.function.name](**args)
            if verbose:
                print(f"  [tool] {tc.function.name}({args}) -> {result}")
            messages.append({"role": "tool", "tool_call_id": tc.id,
                             "content": str(result)})

    return None, messages

In [5]:
answer, messages = run(
    "What time is it in Bangkok, and what is 47281 * 918?"
)
print("\nanswer:", answer)
print("turns :", [m["role"] for m in messages])

  [tool] current_time({'timezone': 'Asia/Bangkok'}) -> 2026-07-16 21:04:58 +07
  [tool] multiply({'a': 47281, 'b': 918}) -> 43403958

answer: The current time in Bangkok is 21:04:58 +07, and the result of the multiplication is 43403958.
turns : ['user', 'assistant', 'tool', 'tool']


## Weaknesses

- **Tool choice is probabilistic.** "What time is it in Bangkok?" on its own returned
  `tool_calls` on some runs and nothing at all on others — and `tool_choice="required"`
  didn't force it. Asked together with the multiplication it reliably calls both. The schema
  is prompt, not a contract.
- **`finish_reason` is `"stop"`, never `"tool_calls"`.** DeepInfra doesn't set the standard
  value, so branching on it silently never fires. Test `message.tool_calls` instead.
- **The raw template leaks into `content`.** Alongside a valid `tool_calls` you get
  `'function=current_time>{"timezone": "Asia/Bangkok"}</function>'`, or sometimes just `'>'`.
  Read `tool_calls`; ignore `content` when it's set.
- **`arguments` is a JSON string, and untrusted.** It needs `json.loads`, and
  `IMPLS[name](**args)` trusts the model completely — a bad name raises `KeyError`, bad keys
  raise `TypeError`, malformed JSON raises too.
- **The `tool` turn must echo `tool_call_id`.** With two calls in one turn (as above) that id
  is the only thing matching each result to its request.